In [ ]:
# ===== Core LangChain abstractions =====
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory

# ===== Community integrations =====
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import TextLoader
from langchain_community.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ===== Hugging Face =====
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline



In [1]:
import langchain_core
import langchain
import langchain_community

print(langchain_core.__version__)
print(langchain.__version__)


1.2.7
1.2.7


In [ ]:
model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id
)


In [ ]:
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.0
)

llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
# Embeddings (Semantic Representation Layer)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:
loader = TextLoader("clinic_docs.txt")
raw_documents = loader.load()


In [ ]:
# Document Ingestion (Knowledge Engineering)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

documents = splitter.split_documents(raw_documents)
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)


In [ ]:
# Vector Store (Semantic Index)

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


In [ ]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
                    You are a professional clinic assistant.

                    POLICY RULES:
                    - Answer ONLY using the provided clinic information.
                    - Do NOT provide medical advice.
                    - Be polite and professional.
                    - If information is missing, respond with:
                    "I will need to check this with the clinic staff."

                    CLINIC KNOWLEDGE:
                    {context}

                    USER QUESTION:
                    {question}

                    ANSWER:
                    """
)


In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    memory=memory
)


In [ ]:
print(qa_chain.run("What are the clinic timings?"))
print(qa_chain.run("Does the doctor specialize in cardiology?"))
print(qa_chain.run("Has the doctor won any awards?"))
print(qa_chain.run("Do you offer emergency consultations?"))
